# 1. Настройка окружения

**Цель:** Проверить установку PyTorch, определить доступные устройства (CPU/GPU/MPS), настроить логирование.

---

In [ ]:
import sys
import logging
import os

LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    stream=sys.stderr,
)
log = logging.getLogger("setup")
log.info("Logging configured at %s level", LOG_LEVEL)

In [ ]:
import torch

log.debug("Importing PyTorch")
print(f"PyTorch version: {torch.__version__}")
log.info("PyTorch %s loaded", torch.__version__)

In [ ]:
import platform

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
log.debug("System info gathered")

In [ ]:
# Проверка CUDA
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    log.info("CUDA device: %s", torch.cuda.get_device_name(0))
else:
    log.warning("CUDA not available — either no NVIDIA GPU or no CUDA toolkit")

In [ ]:
# Проверка MPS (Apple Silicon)
mps_available = torch.backends.mps.is_available()
mps_built = torch.backends.mps.is_built()
print(f"MPS available: {mps_available}")
print(f"MPS built: {mps_built}")
if mps_available:
    log.info("MPS is available — will use Apple Silicon GPU")
else:
    log.warning("MPS not available")

In [ ]:
# Выбор устройства
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Selected device: {device}")
log.info("Using device: %s", device)

In [ ]:
# Тестовый тензор на выбранном устройстве
x = torch.randn(3, 3, device=device)
print(f"Test tensor shape: {x.shape}")
print(f"Test tensor device: {x.device}")
print(f"Test tensor dtype: {x.dtype}")
print(x)
log.debug("Test tensor created on %s: shape=%s", device, x.shape)

In [ ]:
# Производительность: базовый benchmark
import time

sizes = [100, 1000, 5000]
for n in sizes:
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    
    # Прогрев
    for _ in range(5):
        c = a @ b
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    
    # Замер
    start = time.perf_counter()
    for _ in range(20):
        c = a @ b
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / 20
    
    print(f"Matrix multiply {n}x{n}: {elapsed*1000:.2f} ms")
    log.info("Benchmark %dx%d: %.2f ms", n, n, elapsed * 1000)

In [ ]:
print("\n=== Environment check complete ===")
print(f"Summary: PyTorch {torch.__version__} on {device}")
log.info("Environment check complete — ready for transformer study")